# Módulo 4 — Entregable: animación y cierre

Ejercicio 4 del Día 7 (animación) + checkpoint de cierre del módulo (ver
[`README.md`](../README.md)). WRF real (proyecto bigua, Argentina/Uruguay,
2024-12-13, 3 pasos de tiempo cada 2 h).

In [1]:
import xarray as xr
import xwrf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature

raw = xr.open_dataset("../../../data/grid_muestra/wrfout_real_muestra.nc")
ds = raw.xwrf.postprocess().xwrf.destagger()
t2_c = ds["T2"] - 273.15
extent = [float(ds.XLONG.min()), float(ds.XLONG.max()), float(ds.XLAT.min()), float(ds.XLAT.max())]
print("Pasos de tiempo disponibles:", ds.sizes["Time"], "(reales, cada 2 h)")

Pasos de tiempo disponibles: 3 (reales, cada 2 h)


## Animación de temperatura

Una figura por paso de tiempo, con la **misma escala de colores fija** en
todos los cuadros para que la animación sea comparable entre pasos —tal
como se explica en la teoría (Sección 3 del README, Día 7)—. Con solo 3
pasos de tiempo reales, la animación es corta pero suficiente para
demostrar la técnica; con una corrida real completa (varios días) se vería
igual, solo con más cuadros.

In [2]:
vmin, vmax = float(t2_c.min()), float(t2_c.max())

fig = plt.figure(figsize=(6.5, 5.5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent(extent, crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.8)

malla = ax.pcolormesh(
    ds["XLONG"], ds["XLAT"], t2_c.isel(Time=0),
    transform=ccrs.PlateCarree(), cmap="RdYlBu_r", shading="auto",
    vmin=vmin, vmax=vmax,
)
cbar = plt.colorbar(malla, ax=ax, pad=0.05)
cbar.set_label("T2 (°C)")
titulo = ax.set_title("")


def actualizar(i):
    malla.set_array(t2_c.isel(Time=i).values.ravel())
    titulo.set_text(f"T2 real (WRF) — paso {i} ({str(ds['Time'].isel(Time=i).values)[:16]})")
    return malla, titulo


anim = animation.FuncAnimation(fig, actualizar, frames=ds.sizes["Time"], interval=700, blit=False)
anim.save("animacion_t2.gif", writer="pillow", fps=1)
plt.close(fig)
print("Animación guardada: animacion_t2.gif")

Animación guardada: animacion_t2.gif


## Verificación rápida: la animación cubre todo el rango de tiempo

In [3]:
import os

tamano_kb = os.path.getsize("animacion_t2.gif") / 1024
print(f"Tamaño del GIF: {tamano_kb:.1f} KB")
print(f"Cuadros esperados: {ds.sizes['Time']}")
assert tamano_kb > 10, "El GIF parece demasiado pequeño; revisar la generación."
print("OK: el archivo se generó con un tamaño razonable.")

Tamaño del GIF: 278.8 KB
Cuadros esperados: 3
OK: el archivo se generó con un tamaño razonable.


## Checkpoint de cierre del módulo

**Pregunta:** liste las cinco fuentes de error silencioso trabajadas en
este módulo y explique en una frase cómo se evita cada una.

✍️ **Tu respuesta** (doble clic en esta celda y escribe debajo):

…

<details>
<summary>Ver una respuesta modelo (ábrela solo después de escribir la tuya)</summary>

<pre>
1. Grilla staggered (Arakawa-C): U y V estan desplazados medio punto de
   grilla respecto a las variables de masa (T2, etc). Se evita usando
   xwrf.postprocess().xwrf.destagger() antes de comparar o graficar U/V
   junto a variables de masa.

2. Proyeccion incorrecta al graficar: XLAT/XLONG son 2D porque la grilla
   esta en una proyeccion cartografica (Mercator en este dominio real;
   Lambert Conformal en muchos otros dominios), no en lat/lon
   regular. Se evita distinguiendo el "transform" (PlateCarree, como estan
   los datos) del "projection" del eje (como se dibuja el mapa), y dejando
   que xwrf reconstruya el CRS real desde los atributos globales -- xwrf
   reconoce ambas proyecciones automaticamente.

3. Viento grid-relative usado como si fuera earth-relative: U10/V10 crudos
   estan relativos a los ejes de la grilla del modelo, no al norte
   geografico real. Se evita usando wind_east_10/wind_north_10 (calculados
   por xwrf a partir de SINALPHA/COSALPHA) al graficar o comparar contra
   observaciones. (En este dominio real Mercator la diferencia resulto ser
   cero en todo el dominio -- una propiedad real de esa proyeccion, no
   siempre sera asi con otras proyecciones.)

4. Variable acumulada mal interpretada: RAINC/RAINNC son acumulados desde
   el inicio de la simulacion, no un valor por paso de tiempo. Se evita
   calculando SIEMPRE la diferencia entre pasos de tiempo consecutivos
   (precip_total.diff("Time")) antes de interpretar un valor -- y
   verificando el espaciado real entre esos pasos (en este caso, 2 horas,
   no 1).

5. Promedio espacial sin ponderar: un .mean() simple asume que todos los
   puntos de grilla representan la misma area fisica, razonable en
   dominios de mesoescala como este (3 km de resolucion), pero no en
   dominios muy extensos o en latitudes altas.
</pre>
</details>

---
**Fin del Módulo 4.** Siguiente:
[Módulo 5 — Validación de resultados](../../modulo_05_validacion_resultados/README.md).